# Validate Healthcare lineage

Validate the real governed Landing → Bronze → Silver → Gold flow without creating synthetic lineage tables.


In [ ]:
import re
# oidlUtils is injected by AIDP Workbench; no import is required.

def required_parameter(name):
    value = oidlUtils.parameters.getParameter(name, "")
    if value is None or not str(value).strip():
        raise ValueError(f"Missing AIDP job parameter: {name}")
    return str(value).strip()

participant_key = required_parameter("participant_key")
lab_id = required_parameter("lab_id")
workspace_root = required_parameter("workspace_root")
bucket_name = required_parameter("bucket_name")
objectstorage_namespace = required_parameter("objectstorage_namespace")
catalog_name = required_parameter("catalog_name")

participant_match = re.fullmatch(r"u([1-9][0-9]*)", participant_key)
if participant_match is None or int(participant_match.group(1)) < 101:
    raise ValueError("Invalid participant_key")
if lab_id != 'healthcare':
    raise ValueError("This notebook belongs to a different lab")
if not workspace_root.startswith("/Workspace/medallon/"):
    raise ValueError("Invalid workspace_root")
if catalog_name != f"{participant_key}_aidp":
    raise ValueError("Invalid participant catalog")
spark.conf.set("spark.aidp.lineage.enabled", "true")

def table(layer, logical_name):
    prefix = f"{lab_id}_"
    physical_name = logical_name if logical_name.startswith(prefix) else prefix + logical_name
    return f"{catalog_name}.oci_{layer}.{participant_key}_{physical_name}"

expected_tables = {'bronze': ['patients', 'providers', 'appointments', 'encounters'], 'gold': ['healthcare_patient_utilization', 'healthcare_provider_daily'], 'landing': ['patients', 'providers', 'appointments', 'encounters'], 'silver': ['patients', 'providers', 'appointments', 'encounters', 'quality_issues']}
for layer, logical_names in expected_tables.items():
    for logical_name in logical_names:
        target = table(layer, logical_name)
        assert spark.table(target).count() > 0, f"{target} must not be empty"
        details = spark.sql(f"DESCRIBE FORMATTED {target}")
        formatted = {
            str(row["col_name"]).strip().lower(): str(row["data_type"]).strip().lower()
            for row in details.collect()
        }
        assert formatted.get("provider") == "delta", f"{target} must use Delta"
        assert formatted.get("type") == "managed", f"{target} must be managed"

assert spark.conf.get("spark.aidp.lineage.enabled", "true").lower() == "true"
print("healthcare validated across governed Landing, Bronze, Silver and Gold tables")
